In [ ]:
# ============================================
# doc_hint_matching_experiment.ipynb
#
# [실험 목적]
# extract_doc_hints_multi() 함수가 질문에서 맞는 문서를 못 찾거나
# 엉뚱한 문서를 찾는 사례들을 하나씩 재현해서, 정확히 어느 단계
# (기관명 매칭/파일명 유사도/발주기관 내 문서 선택)에서 왜 틀리는지
# 원인을 밝히고 고치는 게 목적
#
# [진행 방식과 알아낸 것]
#
# 1. 행정구역 접미사 노이즈 (cell 12~14)
#    - "인천광역시가 발주한 ..." 질문을 넣었더니 전혀 관계없는
#      "재단법인 광주광역시 광주문화재단"이 문서 힌트로 잡히는 걸 발견
#    - 원인: "광역시"라는 접미사가 COMMON_SUFFIX_WORDS(매칭에서 제외해야
#      할 흔한 단어 목록)에 없어서, "OO광역시"라는 부분 문자열만
#      맞아도 걸려버림
#    - "광역시"/"특별시"/"특별자치시"/"특별자치도"를 목록에 추가하고,
#      normalize_org_name()으로 이 접미사를 뗀 이름끼리 비교하도록 함
#
#
# 2. answer_generation.py 최초 통합본 작성 (cell 18)
#    - 그동안 조각조각 실험하던 코드(문서 힌트 매칭, 법률 키워드 맵,
#      기권 규칙)를 하나의 answer_generation.py 파일로 정리해서 저장
#      -> 이후 실험은 이 파일을 계속 불러와서 수정하는 방식으로 진행
#
#
# 3. fuzzy 매칭이 너무 헐거운 문제 (cell 23, 27, 31~34)
#    - "한국수자원공사가 용인 반도체 산업단지에..." 질문에서 fuzzy_match의
#      최소 겹침 길이(min_overlap=4)가 너무 짧아서 관계없는 부분
#      문자열까지 매칭되는 걸 디버그용 함수를 따로 만들어 한 글자씩
#      추적하며 확인
#    - min_overlap을 6으로 올리기 전에, 기존에 잘 맞던 GKL 케이스가
#      이 값을 올려도 계속 잘 맞는지 먼저 확인한 다음 4->6으로 상향
#
#
# 4. 같은 발주기관 내 여러 문서 선택 로직의 별도 버그 발견 (cell 55~56)
#    - "한국철도공사 운행정보기록 자동분석시스템 개량 사업" 질문에서,
#      "개량"이라는 단어를 3단계(파일명 유사도 매칭)에서만 걸러내니까
#      문제가 해결된 것처럼 보였음
#    - 근데 같은 발주기관(한국철도공사) 안에 문서가 여러 개일 때 그 중
#      하나를 최종적으로 고르는 4단계 로직은 "개량"이라는 단어를 따로
#      체크하지 않고 있어서, 여기서 같은 종류의 오탐이 다시 재발함을
#      확인
#    - 알아낸 것: 같은 단어 하나가 매칭 로직의 여러 단계에 걸쳐 각각
#      쓰이고 있어서, 한 군데만 고치면 "해결된 것처럼 보이지만" 실제로는
#      다른 단계에서 똑같은 문제가 남아있을 수 있다는 걸 실제 재현으로
#      확인함
#
#
# 5. 기타 매칭 케이스 검증 (cell 51, 60, 64~65)
#    - "한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역" 개별 파일명
#      매칭, "축산물품질평가원" 기관명 매칭을 각각 재현해서 문제없는지
#      확인
#    - "다음 6개 사업을 비교해주세요"(c19) 질문에서 "시스템"이라는 짧은
#      2글자 키워드가 매칭 조건을 어떻게 통과하는지 직접 확인
#
# ============================================

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/sprint-public-procurement-rag-assistant
!pwd

/content/sprint-public-procurement-rag-assistant
/content/sprint-public-procurement-rag-assistant


In [3]:
import sys
import types
import src.data_processing.chunking as real_chunking

import pickle

from src.retrieval.indexing import HybridIndex
from src.data_processing.chunking import Chunk

import src.config as config
from pathlib import Path
import src.retrieval.indexing as indexing_module

config.CHROMA_DIR = Path('/content/drive/MyDrive/중급 프로젝트/chroma_db')

indexing_module.CHROMA_DIR = config.CHROMA_DIR
chunking_alias = types.ModuleType('src.chunking')
chunking_alias.Chunk = real_chunking.Chunk
sys.modules['src.chunking'] = chunking_alias

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')
with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)

index = HybridIndex(chunks)

[HybridIndex] parent 전략 chunk 3664개는 검색 후보에서 제외(context 확장 조회 전용) - 실제 검색 대상 14575개
[embeddings] SentenceTransformer 모델 로드 시도 중... (처음 실행이면 HuggingFace에서 모델을 내려받아 몇 분 걸릴 수 있습니다)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

[embeddings] SentenceTransformer 사용: nlpai-lab/KURE-v1 (dim=1024)
[HybridIndex] 기존 임베딩 인덱스 재사용: output/chroma_db (collection=rfp_chunks__nlpai-lab_KURE-v1, backend=nlpai-lab/KURE-v1, 검색 대상 chunk 14575개 일치, 재임베딩 건너뜀)


In [4]:
child_chunks = index._searchable_chunks
print(f"검색 대상(child) chunk 수: {len(child_chunks)}")

검색 대상(child) chunk 수: 14575


In [5]:
from google.colab import userdata
import openai

api_key = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=api_key)

In [6]:
seen = set()
all_filenames_with_biz = []
for c in child_chunks:
    if c.doc_id not in seen:
        seen.add(c.doc_id)
        biz = c.metadata.get('발주_기관', '')
        all_filenames_with_biz.append((c.doc_id, biz))

print(f"고유 문서 수: {len(all_filenames_with_biz)}")

고유 문서 수: 98


In [7]:
# generation_prompts.py, answer_generation.py 불러오기
from generation_prompts import SYSTEM_PROMPT_V9, METADATA_DISTINCTION_INSTRUCTION, needs_metadata_distinction
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

sys.path.append('/content/drive/MyDrive/중급 프로젝트')

print("import 성공")

import 성공


In [8]:
from collections import Counter

org_counts = Counter()
for fname, biz in all_filenames_with_biz:
    org_part = fname.split('_')[0].strip()
    org_counts[org_part] += 1

# 같은 기관에 문서가 2개 이상인 경우
multi_doc_orgs = {org: count for org, count in org_counts.items() if count >= 2}
print(f"문서가 2개 이상인 발주기관: {len(multi_doc_orgs)}개")
for org, count in sorted(multi_doc_orgs.items(), key=lambda x: -x[1]):
    print(f"  {org}: {count}개")

문서가 2개 이상인 발주기관: 11개
  한국수자원공사: 3개
  한국철도공사 (용역): 3개
  한국연구재단: 2개
  한국생산기술연구원: 2개
  인천광역시: 2개
  국방과학연구소: 2개
  수협중앙회: 2개
  한국농어촌공사: 2개
  축산물품질평가원: 2개
  광주과학기술원: 2개
  국민연금공단: 2개


In [9]:
# 각 발주기관별 문서 목록 확인
for org in multi_doc_orgs.keys():
    matching = [f for f, biz in all_filenames_with_biz if f.startswith(org)]
    print(f"{org}")
    for m in matching:
        print(f"  {m}")
    print()

한국연구재단
  한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp
  한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp

한국생산기술연구원
  한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp
  한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp

인천광역시
  인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp
  인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp
  인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp

국방과학연구소
  국방과학연구소_대용량 자료전송시스템 고도화.hwp
  국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp

한국수자원공사
  한국수자원공사_건설통합시스템(CMS) 고도화.hwp
  한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp
  한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp

한국철도공사 (용역)
  한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp
  한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp
  한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp

수협중앙회
  수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp
  수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp

한국농어촌공사
  한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp
  한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp

축산물품질평가원
  축산물품질평가원_축산물이력관리시스템 개선(정보화 사업).hwp
  축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp

광주과학기술원
  광주과학기술원_학사시스템 기능개선 사업.hwp
  광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 

In [10]:
# 인천광역시 케이스 먼저 확인 - "인천광역시"와 "인천광역시 동구"가 실제로 구분되는지
q = "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?"
doc_hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print("문서 힌트:", doc_hints)

문서 힌트: ['인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp', '인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp', '재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp', '인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp']


In [11]:
# 각 발주기관별로 "질문에 없는 핵심 단어" 기반 테스트 질문 만들어서 확인
test_cases = [
    ("한국생산기술연구원", "한국생산기술연구원이 진행하는 고압가스 관련 안전관리시스템 구축 용역의 사업기간은?"),
    ("국방과학연구소", "국방과학연구소의 대용량 데이터를 옮기는 시스템을 개선하는 사업 예산은?"),
    ("한국수자원공사", "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?"),
    ("수협중앙회", "수협중앙회가 강릉 지역 어선의 안전 조업을 관리하는 시스템 구축 사업기간은?"),
    ("축산물품질평가원", "축산물품질평가원이 가축 이력을 관리하는 시스템을 개선하는 사업의 예산은?"),
    ("광주과학기술원", "광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?"),
]

for org, q in test_cases:
    doc_hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{org}] {q}")
    print(f"  힌트: {doc_hints}")
    print()

[한국생산기술연구원] 한국생산기술연구원이 진행하는 고압가스 관련 안전관리시스템 구축 용역의 사업기간은?
  힌트: ['한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp', '한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp']

[국방과학연구소] 국방과학연구소의 대용량 데이터를 옮기는 시스템을 개선하는 사업 예산은?
  힌트: ['국방과학연구소_대용량 자료전송시스템 고도화.hwp', '국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp']

[한국수자원공사] 한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?
  힌트: ['한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp', '한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp', '한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp', '한국수자원공사_건설통합시스템(CMS) 고도화.hwp', '한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp']

[수협중앙회] 수협중앙회가 강릉 지역 어선의 안전 조업을 관리하는 시스템 구축 사업기간은?
  힌트: ['수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp', '수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp']

[축산물품질평가원] 축산물품질평가원이 가축 이력을 관리하는 시스템을 개선하는 사업의 예산은?
  힌트: ['축산물품질평가원_축산물이력관리시스템 개선(정보화 사업).hwp', '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp']

[광주과학기술원] 광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?
  힌트: ['광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp', '재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp', '광주과학기술원_학사시스템 기능개선 사업.hwp']



In [12]:
q = "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?"

# 어느 단계에서 이 엉뚱한 문서들이 잡히는지 확인
for fname, biz in all_filenames_with_biz:
    if '광주광역시' in fname or '한국산업단지공단' in fname:
        org_part = fname.split('_')[0].strip()
        print(f"파일명: {fname[:50]}")
        print(f"  org_part: {org_part}")

파일명: 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp
  org_part: 한국산업단지공단
파일명: 재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp
  org_part: 재단법인 광주광역시 광주문화재단


In [13]:
import re

def normalize_org_name(name):
    return re.sub(r'(특별시|광역시|특별자치시|특별자치도)', '', name)

q1 = "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?"
q2 = "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?"

test_orgs = ['인천광역시', '재단법인 광주광역시 광주문화재단', '한국산업단지공단', '한국수자원조사기술원']

for org_part in test_orgs:
    org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
    org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
    org_core_norm = normalize_org_name(org_core_clean)
    print(f"org_part: '{org_part}'")
    print(f"  org_core_clean: '{org_core_clean}'")
    print(f"  org_core_norm: '{org_core_norm}'")
    print(f"  q1에 org_core_norm 포함?: {org_core_norm in q1}")
    print(f"  q2에 org_core_norm 포함?: {org_core_norm in q2}")
    print()

org_part: '인천광역시'
  org_core_clean: '인천광역시'
  org_core_norm: '인천'
  q1에 org_core_norm 포함?: True
  q2에 org_core_norm 포함?: False

org_part: '재단법인 광주광역시 광주문화재단'
  org_core_clean: '재단법인 광주광역시 광주문화재단'
  org_core_norm: '재단법인 광주 광주문화재단'
  q1에 org_core_norm 포함?: False
  q2에 org_core_norm 포함?: False

org_part: '한국산업단지공단'
  org_core_clean: '한국산업단지공단'
  org_core_norm: '한국산업단지공단'
  q1에 org_core_norm 포함?: False
  q2에 org_core_norm 포함?: False

org_part: '한국수자원조사기술원'
  org_core_clean: '한국수자원조사기술원'
  org_core_norm: '한국수자원조사기술원'
  q1에 org_core_norm 포함?: False
  q2에 org_core_norm 포함?: False



In [14]:
COMMON_SUFFIX_WORDS = {
    '박물관', '시스템', '센터', '공단', '진흥원', '협회', '재단', '연구원', '공사', '대학교',
    '사업', '관리', '운영', '구축', '개선', '개발', '지원', '정보', '용역', '기관', '기술',
    '고도화', '확대', '기능', '서비스', '일자리', '플랫폼', '통합', '접수',
    '일자리재단', '일자리플랫폼', '보험', '입찰공고', '공고',
    '과학연구', '과학연', '학연구', '연구소', '기록관리', '경기기록',
    '학교', '학교 ', ' 학교', '산학협력단', '산학협력', '학협력단',
    '통합시스템',
    '2024년', '2025년',
}

q1 = "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?"
q2 = "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?"

def find_fuzzy_match(target_str, question, min_len=4):
    for start in range(len(target_str) - min_len + 1):
        for length in range(len(target_str) - start, min_len - 1, -1):
            substr = target_str[start:start+length]
            if substr.strip() in question and substr.strip() not in COMMON_SUFFIX_WORDS:
                return substr.strip()
    return None

test_cases = [
    ('재단법인 광주 광주문화재단', q1),
    ('한국산업단지공단', q2),
]

for org_core_norm, q in test_cases:
    match = find_fuzzy_match(org_core_norm, q)
    print(f"'{org_core_norm}' vs 질문 -> 매칭된 부분문자열: '{match}'")

'재단법인 광주 광주문화재단' vs 질문 -> 매칭된 부분문자열: 'None'
'한국산업단지공단' vs 질문 -> 매칭된 부분문자열: '산업단지'


In [15]:
q1 = "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?"

COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}

stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', q1) if len(w) >= 4]
keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]
print("3단계 키워드:", keywords_all)

def fuzzy_match(kw, text, min_overlap=4):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

target_fname = '재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp'
fname_clean = target_fname.replace('.hwp', '')
matched_kws = [kw for kw in keywords_all if fuzzy_match(kw, fname_clean)]
print(f"이 파일에 매칭된 키워드: {matched_kws}")

3단계 키워드: ['인천광역시가', '도시계획위원회', '통합관리시스템은', '사업기간이']
이 파일에 매칭된 키워드: []


In [16]:
q1 = "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?"

org_part = '재단법인 광주광역시 광주문화재단'
org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
org_core_norm = normalize_org_name(org_core_clean)

print(f"org_core_clean: '{org_core_clean}'")
print(f"org_core_norm: '{org_core_norm}'")

# 1단계 fuzzy 매칭 로직 그대로 재현
min_len = 4
matched = False
matched_substr = None
for target_str in [org_core_clean, org_core_norm]:
    for start in range(len(target_str) - min_len + 1):
        for length in range(len(target_str) - start, min_len - 1, -1):
            substr = target_str[start:start+length]
            if substr.strip() in q1 and substr.strip() not in COMMON_SUFFIX_WORDS:
                matched = True
                matched_substr = substr.strip()
                break
        if matched:
            break
    if matched:
        break

print(f"매칭됨: {matched}, 매칭된 부분: '{matched_substr}'")

org_core_clean: '재단법인 광주광역시 광주문화재단'
org_core_norm: '재단법인 광주 광주문화재단'
매칭됨: True, 매칭된 부분: '광역시'


In [17]:
COMMON_SUFFIX_WORDS_V2 = COMMON_SUFFIX_WORDS | {'광역시', '특별시', '특별자치시', '특별자치도'}

matched = False
matched_substr = None
org_core_clean = '재단법인 광주광역시 광주문화재단'
min_len = 4
for target_str in [org_core_clean]:
    for start in range(len(target_str) - min_len + 1):
        for length in range(len(target_str) - start, min_len - 1, -1):
            substr = target_str[start:start+length]
            if substr.strip() in q1 and substr.strip() not in COMMON_SUFFIX_WORDS_V2:
                matched = True
                matched_substr = substr.strip()
                break
        if matched:
            break
    if matched:
        break

print(f"매칭됨: {matched}, 매칭된 부분: '{matched_substr}'")

매칭됨: False, 매칭된 부분: 'None'


In [23]:
%%writefile "/content/drive/MyDrive/중급 프로젝트/answer_generation.py"
# ============================================
# answer_generation.py
# RFP 질문에 대해 문서 힌트 추출 -> 조건 필터링 -> 컨텍스트 구성 -> LLM 답변 생성까지
# 담당하는 최종 답변 생성 파이프라인
#
# [주의] 이 import 경로는 드라이브 단독 테스트용입니다.
#        실제 레포(src/generation/)에 반영할 때는
#        "from generation_prompts import" 를
#        "from src.generation.generation_prompts import" 로 바꿔야 합니다.
#
# 개발 히스토리:
# - extract_doc_hints_multi (v1): 기관명/사업명/파일명 유사도 매칭 3단계로
#   문서 힌트를 찾고, 같은 발주기관에 문서가 여러 개면 질문 키워드와 파일명을
#   정확 매칭해서 하나를 선택
# - extract_doc_hints_multi_v2: 같은 발주기관 내 문서 선택 단계에서, 질문에
#   포함된 발주기관명이나 "용역은/사업은" 같은 흔한 조사형 단어가 모든 후보
#   문서에 동일하게 매칭되면서 노이즈로 작용해 정답 판별 신호(예: "운행기록"
#   ↔ 파일명의 "운행정보기록")가 묻히는 문제 발견. 해당 단어들을 stopwords에
#   추가하고, 조사를 뗀 뒤 부분 문자열로 매칭하도록 완화해 해결.
#   core40 40문항 전체 회귀 검증 결과 영향 0건, rag-56은 목표 문항(c09) 1건만
#   정확히 개선되고 나머지 55건은 변화 없음을 확인.
# - COMMON_SUFFIX_WORDS 보강: 1단계(기관명 fuzzy 매칭)에서 "광역시", "특별시"
#   같은 행정구역 접미사가 블랙리스트에 없어, "인천광역시"를 물으면 전혀 무관한
#   "OO광역시"라는 이름의 다른 기관까지 매칭되는 문제 발견(예: 인천광역시 질문에
#   "재단법인 광주광역시 광주문화재단"이 잘못 포함됨). 해당 접미사들을
#   COMMON_SUFFIX_WORDS에 추가해 해결.
# - ask_rfp_v9: keyword_chunks만 쓰던 기존 로직에, 질문이 여러 항목을
#   물을 때 법률 키워드에 안 걸리는 항목이 통째로 누락되는 문제를 발견해
#   문서 앞쪽 청크(사업개요/범위가 보통 위치)를 함께 포함하도록 개선
# ============================================

import re
from generation_prompts import (
    SYSTEM_PROMPT_V9,
    METADATA_DISTINCTION_INSTRUCTION,
    needs_metadata_distinction,
)


ORG_ALIAS_MAP = {
    '대검찰청': ['검찰'],
    '고려대학교': ['고려대'],
    '한국산업단지공단': ['산단'],
    '그랜드코리아레저': ['GKL'],
}

COMMON_SUFFIX_WORDS = {
    '박물관', '시스템', '센터', '공단', '진흥원', '협회', '재단', '연구원', '공사', '대학교',
    '사업', '관리', '운영', '구축', '개선', '개발', '지원', '정보', '용역', '기관', '기술',
    '고도화', '확대', '기능', '서비스', '일자리', '플랫폼', '통합', '접수',
    '일자리재단', '일자리플랫폼', '보험', '입찰공고', '공고',
    '과학연구', '과학연', '학연구', '연구소', '기록관리', '경기기록',
    '학교', '학교 ', ' 학교', '산학협력단', '산학협력', '학협력단',
    '통합시스템',
    '2024년', '2025년',
    '광역시', '특별시', '특별자치시', '특별자치도',
}
COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}

LEGAL_KEYWORDS_MAP = {
    '하도급': ['하도급'], '공동수급': ['공동수급', '지분율', '컨소시엄'], '지분율': ['지분율', '공동수급'],
    '계약보증금': ['계약보증금', '보증금'], '평가': ['배점', '평가비율', '기술평가', '가격평가'],
    '제안서 보상': ['제안서 보상'], '불이익': ['부정당업자', '입찰보증금', '귀속'],
    '제출물': ['제출서류', '부', 'USB', '제출규격'], '제출': ['제출서류', 'USB'], '수량': ['부', 'USB'],
    '구축기간': ['사업기간', '구축기간', '개월'], '사업기간': ['사업기간', '구축기간', '개월'],
    '유지보수': ['무상유지보수', '유지보수기간', '하자보수', '무상 하자보수'], '참가자격': ['참가자격', '참가 자격'],
    '유지관리': ['하자보수', '유지관리 인력', '무상 하자보수'],
    '교육 의무': ['유지관리 인력', '사용자 및 관리자', '하자보수'], '교육을': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '검수 후': ['하자보수', '유지관리 인력'], '재입찰': ['재입찰', '재공고입찰', '최초의 입찰'],
    '재공고': ['재입찰', '재공고입찰', '최초의 입찰'], '조건 변경': ['재입찰', '재공고입찰', '최초의 입찰'],
    '지역 요건': ['주된 영업소', '소재지'], '부산에': ['주된 영업소', '소재지'],
    '지역요건': ['주된 영업소', '소재지'], '소재지': ['주된 영업소', '소재지'],
    '보유인력': ['보유인력', '배점한도'], '배점한도': ['보유인력', '배점한도'],
    '계량평가': ['보유인력', '배점한도', '재무구조'], '규모비율': ['규모비율', '환산점수', '점수비중'],
    '환산점수': ['규모비율', '환산점수', '점수비중'], '수행실적': ['규모비율', '환산점수', '수행실적'],
    '신인도': ['신인도', '가점'], '가점표': ['신인도', '가점'],
    '연구원 승인': ['Lesson', '회람'], '발생한 경우': ['Lesson', '회람'], '회람': ['Lesson', '회람'],
}


def find_relevant_keywords(question):
    matched = []
    for trigger, kws in LEGAL_KEYWORDS_MAP.items():
        if trigger in question:
            matched.extend(kws)
    return list(set(matched))


def is_aggregation_question(question):
    keywords = ['몇 개', '개수', '다 나열', '몇 건']
    strong_total = '전부' in question or ('총' in question and ('개' in question or '건' in question))
    return any(kw in question for kw in keywords) or strong_total


def extract_filter_conditions(query):
    conditions = {}
    if '억' in query and ('이상' in query or '넘는' in query):
        match = re.search(r'(\d+)억', query)
        if match:
            conditions['금액_최소'] = int(match.group(1)) * 100000000
    if '지자체' in query or '지방자치단체' in query:
        conditions['지자체'] = True
    if '공사' in query and ('OO공사' in query or '발주기관이' in query):
        conditions['공사'] = True
    if 'AI' in query:
        conditions['주제_AI'] = True
    if '긴급' in query:
        conditions['긴급'] = True
    if '보안' in query:
        conditions['보안'] = True
    if '재난' in query:
        conditions['재난'] = True
    return conditions


def is_local_gov(org):
    if org is None or (isinstance(org, float)):
        return False
    return bool(re.search(r'(광역시|특별시|특별자치도|특별자치시|[가-힣]+도|[가-힣]+시|[가-힣]+군|[가-힣]+구)$', str(org).strip()))


def normalize_org_name(name):
    return re.sub(r'(특별시|광역시|특별자치시|특별자치도)', '', name)


def extract_doc_hints_multi(question, all_filenames_with_biz):
    q_no_space = question.replace(' ', '').replace('&', '')
    org_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        org_part = fname.replace('refined_', '').split('_')[0].strip()
        org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
        org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
        org_core_clean = re.sub(r'\s*입찰공고\s*$', '', org_core_clean).strip()
        org_core_norm = normalize_org_name(org_core_clean)
        if len(org_core_clean) < 2:
            continue

        matched = False
        if org_core_clean in question:
            matched = True
        elif len(org_core_norm) >= 3 and org_core_norm in question:
            matched = True
        elif org_core_clean in ORG_ALIAS_MAP and any(alias in question for alias in ORG_ALIAS_MAP[org_core_clean]):
            matched = True
        else:
            min_len = 4
            for target_str in [org_core_clean, org_core_norm]:
                for start in range(len(target_str) - min_len + 1):
                    for length in range(len(target_str) - start, min_len - 1, -1):
                        substr = target_str[start:start+length]
                        if substr.strip() in question and substr.strip() not in COMMON_SUFFIX_WORDS:
                            matched = True
                            break
                    if matched:
                        break
                if matched:
                    break
        if matched:
            org_candidates.append((fname, org_core_clean))

    biz_candidates = []
    quoted = re.findall(r"['\"]([^'\"]+)['\"]", question)
    for fname, biz_name in all_filenames_with_biz:
        biz_name = str(biz_name).strip()
        if len(biz_name) >= 4 and biz_name in question:
            biz_candidates.append(fname)
            continue
        for q_ in quoted:
            if q_ in biz_name or biz_name in q_:
                biz_candidates.append(fname)
                break
        eng_words = re.findall(r'[A-Za-z][A-Za-z&\s]{2,}[A-Za-z]', biz_name)
        for ew in eng_words:
            ew_no_space = ew.strip().replace(' ', '').replace('&', '')
            if len(ew_no_space) >= 4 and ew_no_space in q_no_space:
                biz_candidates.append(fname)
                break

    stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
    raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', question) if len(w) >= 4]
    keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]

    def fuzzy_match(kw, text, min_overlap=4):
        kw_ns = kw.replace(' ', '')
        text_ns = text.replace(' ', '')
        if kw_ns in text_ns:
            return True
        for n in range(len(kw_ns), min_overlap - 1, -1):
            if kw_ns[:n] in text_ns:
                return True
        return False

    def keyword_weight(kw):
        return 3 if re.search(r'[A-Za-z]', kw) else 1

    filename_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        fname_clean = fname.replace('refined_', '').replace('.hwp', '').replace('.pdf', '')
        matched_kws = [kw for kw in keywords_all if fuzzy_match(kw, fname_clean)]
        score = sum(keyword_weight(kw) for kw in matched_kws)
        if score > 0:
            filename_candidates.append((fname, score, len(matched_kws)))

    if filename_candidates:
        filename_candidates.sort(key=lambda x: -x[1])
        max_score = filename_candidates[0][1]
        for top_fname, score, cnt in filename_candidates:
            if score >= max_score * 0.6 or score >= 1:
                if top_fname not in [f for f, _ in org_candidates] and top_fname not in biz_candidates:
                    if len(filename_candidates) <= 3 or score >= max(max_score * 0.6, 1):
                        biz_candidates.append(top_fname)

    org_groups = {}
    for fname, org_core in org_candidates:
        org_groups.setdefault(org_core, []).append(fname)

    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교',
                 '용역은', '용역이', '용역을', '사업은', '사업이', '사업을'}
    keywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords]

    def _fuzzy_kw_match(kw, text):
        kw_clean = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', kw)
        if len(kw_clean) < 2:
            return False
        return kw_clean in text.replace(' ', '')

    final_hints = []
    for org_core, fnames in org_groups.items():
        fnames = list(set(fnames))
        if len(fnames) == 1:
            final_hints.append(fnames[0])
        else:
            fname_to_biz = dict(all_filenames_with_biz)
            best_doc, best_score2 = None, -1
            for fname in fnames:
                biz_name = fname_to_biz.get(fname, '')
                score2 = sum(1 for kw in keywords if _fuzzy_kw_match(kw, fname) or _fuzzy_kw_match(kw, str(biz_name)))
                if score2 > best_score2:
                    best_score2, best_doc = score2, fname
            final_hints.append(best_doc)

    for fname in biz_candidates:
        if fname not in final_hints:
            final_hints.append(fname)

    return list(dict.fromkeys(final_hints))


def meta_header_from_metadata(doc_id, metadata):
    org = metadata.get('발주_기관', '')
    amt = metadata.get('사업_금액')
    amt_str = f"{amt:,.0f}원" if amt not in (None, '') else "확인되지 않음"
    return f"[문서: {doc_id}]\n[발주기관(메타데이터): {org}]\n[사업금액(메타데이터): {amt_str}]"


def ask_rfp_v9(question, client, index, child_chunks, all_filenames_with_biz,
               model_name="gpt-5-mini", max_retries=2):
    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]
    keywords = find_relevant_keywords(question)
    conditions = extract_filter_conditions(question)

    doc_to_meta = {}
    for c in child_chunks:
        if c.doc_id not in doc_to_meta:
            doc_to_meta[c.doc_id] = c.metadata

    context_parts = []

    def get_doc_chunks(doc_id):
        return [c for c in child_chunks if c.doc_id == doc_id]

    def build_meta_filter(conds):
        if not conds:
            return None
        def _filter(meta):
            if '금액_최소' in conds:
                amt = meta.get('사업_금액')
                if amt is None or amt < conds['금액_최소']:
                    return False
            if conds.get('지자체'):
                if not is_local_gov(meta.get('발주_기관')):
                    return False
            if conds.get('공사'):
                org = str(meta.get('발주_기관', ''))
                if '공사' not in org:
                    return False
            return True
        return _filter

    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0], -1
        for fname in doc_hints:
            biz = doc_to_meta.get(fname, {}).get('발주_기관', '')
            score = sum(1 for kw in qkeywords if kw in fname or kw in str(biz))
            if score > best_score:
                best_score, best_doc = score, fname
        doc_hint = best_doc
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in get_doc_chunks(doc_hint):
            context_parts.append(f"{header}\n{c.text}")

    elif len(doc_hints) == 1 and keywords:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        keyword_chunks = [c for c in doc_c if any(kw in c.text for kw in keywords)]
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        if keyword_chunks:
            combined = keyword_chunks[:20] + doc_c[:10]
            seen_ids = set()
            for c in combined:
                if c.chunk_id in seen_ids:
                    continue
                seen_ids.add(c.chunk_id)
                context_parts.append(f"{header}\n{c.text}")
        else:
            hits = index.hybrid_search(question, k=10, expand_to_parent=True)
            for h in hits:
                context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    elif len(doc_hints) >= 2:
        for doc_hint in doc_hints:
            doc_c = get_doc_chunks(doc_hint)
            if keywords:
                matched = [c for c in doc_c if any(kw in c.text for kw in keywords)]
                selected = matched[:8] if matched else doc_c[:8]
            else:
                selected = doc_c[:8]
            header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
            for c in selected:
                context_parts.append(f"{header}\n{c.text}")

    elif doc_hints:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in doc_c[:15]:
            context_parts.append(f"{header}\n{c.text}")

    elif conditions:
        meta_filter = build_meta_filter(conditions)
        hits = index.hybrid_search(question, k=80, meta_filter=meta_filter, expand_to_parent=True)
        for h in hits:
            context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    else:
        hits = index.hybrid_search(question, k=10, expand_to_parent=True)
        for h in hits:
            context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    context = "\n\n---\n\n".join(context_parts)

    prompt_base = SYSTEM_PROMPT_V9
    if needs_metadata_distinction(question):
        prompt_base = prompt_base.replace(
            "## 컨텍스트 (검색된 문서 조각)",
            METADATA_DISTINCTION_INSTRUCTION + "\n## 컨텍스트 (검색된 문서 조각)"
        )

    final_prompt = prompt_base.format(context=context, question=question)

    for attempt in range(max_retries):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": final_prompt}],
            max_completion_tokens=8000,
            reasoning_effort="low",
        )
        answer = response.choices[0].message.content
        if answer:
            return answer
    return "(답변 생성 실패)"

Overwriting /content/drive/MyDrive/중급 프로젝트/answer_generation.py


In [24]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_doc_hints_multi, COMMON_SUFFIX_WORDS

importlib.reload(answer_generation)

print('광역시' in COMMON_SUFFIX_WORDS)

True


In [25]:
# 아까 문제였던 케이스들 재확인
test_cases = [
    "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?",
    "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?",
    "광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?",
]

for q in test_cases:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?]
  힌트: ['인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp', '인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp', '인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp']

[한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?]
  힌트: ['한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp', '한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp', '한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp', '한국수자원공사_건설통합시스템(CMS) 고도화.hwp', '한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp']

[광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?]
  힌트: ['광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp', '재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp', '광주과학기술원_학사시스템 기능개선 사업.hwp']



In [26]:
q = "광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?"

org_part = '재단법인 광주광역시 광주문화재단'
org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
org_core_norm = normalize_org_name(org_core_clean)
print(f"org_core_clean: '{org_core_clean}'")
print(f"org_core_norm: '{org_core_norm}'")

min_len = 4
for target_str in [org_core_clean, org_core_norm]:
    for start in range(len(target_str) - min_len + 1):
        for length in range(len(target_str) - start, min_len - 1, -1):
            substr = target_str[start:start+length]
            if substr.strip() in q and substr.strip() not in COMMON_SUFFIX_WORDS:
                print(f"매칭됨: '{substr.strip()}'")

org_core_clean: '재단법인 광주광역시 광주문화재단'
org_core_norm: '재단법인 광주 광주문화재단'
매칭됨: '광주'


In [27]:
# 원본 로직 다시 정확히 추적
target_str = '재단법인 광주 광주문화재단'  # org_core_norm
q = "광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?"
min_len = 4

for start in range(len(target_str) - min_len + 1):
    for length in range(len(target_str) - start, min_len - 1, -1):
        substr = target_str[start:start+length]
        if substr.strip() in q and substr.strip() not in COMMON_SUFFIX_WORDS:
            print(f"start={start}, length={length}, substr='{substr}' (strip 후: '{substr.strip()}', len={len(substr.strip())})")

start=4, length=4, substr=' 광주 ' (strip 후: '광주', len=2)


In [28]:
def _debug_fuzzy_org_match(target_str, question, min_len=4):
    for start in range(len(target_str) - min_len + 1):
        for length in range(len(target_str) - start, min_len - 1, -1):
            substr = target_str[start:start+length]
            stripped = substr.strip()
            # strip 후 길이도 min_len 이상인지 재확인 (기존엔 이 체크가 없었음)
            if len(stripped) >= min_len and stripped in question and stripped not in COMMON_SUFFIX_WORDS:
                return stripped
    return None

target_str = '재단법인 광주 광주문화재단'
q = "광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?"
result = _debug_fuzzy_org_match(target_str, q)
print(f"매칭 결과: {result}")

매칭 결과: None


In [29]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = '''                    if substr.strip() in question and substr.strip() not in COMMON_SUFFIX_WORDS:
                            matched = True
                            break'''

new_code = '''                    stripped_substr = substr.strip()
                        if len(stripped_substr) >= min_len and stripped_substr in question and stripped_substr not in COMMON_SUFFIX_WORDS:
                            matched = True
                            break'''

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

수정 완료


In [30]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

importlib.reload(answer_generation)

# 아까 문제였던 케이스 재확인
test_cases = [
    "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?",
    "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?",
    "광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?",
]

for q in test_cases:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?]
  힌트: ['인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp', '인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp', '인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp']

[한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?]
  힌트: ['한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp', '한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp', '한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp', '한국수자원공사_건설통합시스템(CMS) 고도화.hwp', '한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp']

[광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?]
  힌트: ['광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp', '광주과학기술원_학사시스템 기능개선 사업.hwp']



In [31]:
q = "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?"

for org_part in ['한국산업단지공단', '한국수자원조사기술원']:
    org_core_clean = org_part
    org_core_norm = normalize_org_name(org_core_clean)

    min_len = 4
    for target_str in [org_core_clean, org_core_norm]:
        for start in range(len(target_str) - min_len + 1):
            for length in range(len(target_str) - start, min_len - 1, -1):
                substr = target_str[start:start+length]
                stripped = substr.strip()
                if len(stripped) >= min_len and stripped in q and stripped not in COMMON_SUFFIX_WORDS:
                    print(f"[{org_part}] 매칭: '{stripped}'")

[한국산업단지공단] 매칭: '산업단지'
[한국산업단지공단] 매칭: '산업단지'
[한국수자원조사기술원] 매칭: '한국수자원'
[한국수자원조사기술원] 매칭: '한국수자'
[한국수자원조사기술원] 매칭: '국수자원'
[한국수자원조사기술원] 매칭: '한국수자원'
[한국수자원조사기술원] 매칭: '한국수자'
[한국수자원조사기술원] 매칭: '국수자원'


In [32]:
def _debug_fuzzy_org_match_v2(target_str, question, min_len):
    for start in range(len(target_str) - min_len + 1):
        for length in range(len(target_str) - start, min_len - 1, -1):
            substr = target_str[start:start+length]
            stripped = substr.strip()
            if len(stripped) >= min_len and stripped in question and stripped not in COMMON_SUFFIX_WORDS:
                return stripped
    return None

q = "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?"

for min_len_test in [4, 5, 6, 7]:
    result = _debug_fuzzy_org_match_v2('한국수자원조사기술원', q, min_len_test)
    print(f"min_len={min_len_test}: {result}")

min_len=4: 한국수자원
min_len=5: 한국수자원
min_len=6: None
min_len=7: None


In [33]:
# min_len=6으로 다른 케이스들도 재확인
gkl_q = "GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?"
gkl_hints_before = extract_doc_hints_multi(gkl_q, all_filenames_with_biz)
print("GKL 케이스(현재 min_len=4):", gkl_hints_before)

GKL 케이스(현재 min_len=4): ['그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp']


In [34]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "            min_len = 4\n            for target_str in [org_core_clean, org_core_norm]:"
new_code = "            min_len = 6\n            for target_str in [org_core_clean, org_core_norm]:"

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [36]:
test_cases = [
    "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?",
    "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?",
    "광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?",
    "한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?",
]

for q in test_cases:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?]
  힌트: ['인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp', '인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp']

[한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?]
  힌트: ['한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp', '한국수자원공사_건설통합시스템(CMS) 고도화.hwp', '한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp', '한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp', '한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp']

[광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?]
  힌트: ['광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp', '광주과학기술원_학사시스템 기능개선 사업.hwp']

[한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?]
  힌트: ['한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']



In [37]:
q = "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?"

COMMON_FILENAME_WORDS_test = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}
stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', q) if len(w) >= 4]
keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS_test and '입찰공고' not in w]
print("3단계 키워드:", keywords_all)

def fuzzy_match(kw, text, min_overlap=4):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

for fname in ['한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp', '한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp']:
    fname_clean = fname.replace('.hwp', '')
    matched_kws = [kw for kw in keywords_all if fuzzy_match(kw, fname_clean)]
    print(f"{fname}: matched={matched_kws}")

3단계 키워드: ['한국수자원공사가', '산업단지에', '공급하는', '예산은']
한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp: matched=['산업단지에']
한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp: matched=['한국수자원공사가']


In [38]:
def _debug_fuzzy_match_v2(kw, text, min_overlap):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

for min_overlap_test in [4, 5, 6]:
    r1 = _debug_fuzzy_match_v2('산업단지에', '한국산업단지공단_산단 안전정보시스템 1차 구축 용역', min_overlap_test)
    r2 = _debug_fuzzy_match_v2('한국수자원공사가', '한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계', min_overlap_test)
    print(f"min_overlap={min_overlap_test}: 산업단지공단={r1}, 수자원조사기술원={r2}")

min_overlap=4: 산업단지공단=True, 수자원조사기술원=True
min_overlap=5: 산업단지공단=False, 수자원조사기술원=True
min_overlap=6: 산업단지공단=False, 수자원조사기술원=False


In [39]:
# GKL 케이스와 다른 core40 문항들에 min_overlap 변경이 영향 없는지 확인
gkl_q = "GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?"

def _debug_fuzzy_match_v3(kw, text, min_overlap):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

# GKL은 영문 약어라 keyword_weight가 3배로 가중치 붙는 케이스 - 확인
test_kw = 'GKL'
test_fname = '그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역'
for mo in [4, 5, 6]:
    r = _debug_fuzzy_match_v3(test_kw, test_fname, mo)
    print(f"min_overlap={mo}: GKL 매칭 = {r}")

min_overlap=4: GKL 매칭 = True
min_overlap=5: GKL 매칭 = True
min_overlap=6: GKL 매칭 = True


In [40]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "    def fuzzy_match(kw, text, min_overlap=4):"
new_code = "    def fuzzy_match(kw, text, min_overlap=6):"

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [41]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

importlib.reload(answer_generation)

In [42]:
# core40 전체 문서 힌트 재확인
import json
from pathlib import Path

DATA_DIR2 = Path('/content/drive/MyDrive/중급 프로젝트')
with open(DATA_DIR2 / 'dev.refined.review-candidate.jsonl', encoding='utf-8') as f:
    core40 = [json.loads(l) for l in f if l.strip()]

with open(DATA_DIR2 / 'rag-56.draft.jsonl', encoding='utf-8') as f:
    rag56 = [json.loads(l) for l in f if l.strip()]

print(f"core40: {len(core40)}개, rag56: {len(rag56)}개")

core40: 40개, rag56: 56개


In [43]:
# core40 전체 문서 힌트 확인
for item in core40:
    question = item['question']
    if item['task_type'] == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    hints = extract_doc_hints_multi(combined_q, all_filenames_with_biz)
    print(f"[{item['case_id']}] 힌트 개수: {len(hints)} - {hints}")

[dev-single-001] 힌트 개수: 1 - ['(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp']
[dev-single-002] 힌트 개수: 1 - ['한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp']
[dev-single-003] 힌트 개수: 1 - ['국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp']
[dev-single-004] 힌트 개수: 1 - ['한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp']
[dev-single-005] 힌트 개수: 1 - ['서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp']
[dev-single-006] 힌트 개수: 3 - ['파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp', '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp', '세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp']
[dev-single-007] 힌트 개수: 1 - ['서울특별시 여성가족재단_(재공고, 협상) 서울 디지털성범죄 안심지원센.hwp']
[dev-single-008] 힌트 개수: 1 - ['사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp']
[dev-single-009] 힌트 개수: 1 - ['국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp']
[dev-single-010] 힌트 개수: 1 - ['서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp']
[dev-multi-001] 힌트 개수: 2 - ['대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp', '한국교육과정평가원_국가교육과정정보센터(NCIC) 시스템 운영 및 개선.hwp']
[dev-multi-002] 힌트 개수: 2 - ['한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp', '파주도시관광공사_

In [44]:
for cid in ['dev-single-006', 'dev-multi-010', 'dev-followup-009']:
    item = next(it for it in core40 if it['case_id'] == cid)
    question = item['question']
    if item['task_type'] == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(f"[{cid}] {question}")
    print(answer)
    print()

[dev-single-006] 종량제봉투 판매관리 전산시스템 개선사업에서 구현해야 할 핵심 기능과 기존 데이터 이관 범위는?
핵심 기능(구현해야 할 주요 기능)
- 전산관리 시스템 구축: 주문·판매 내역 조회, 재고관리, 결제내역 관리, 보고서식 및 통계자료 조회 기능  
- 인터넷 주문 프로그램 연동: 온라인 주문·주문내역 확인 기능 및 다양한 디바이스(PC/모바일/태블릿) 지원(반응형 웹)  
- 결제시스템 개발 및 연동: 카드결제 및 계좌이체(가상계좌) 연동(PG사 API 연동 포함)  
- 관리자 기능 연계: 기존 관리자 기능(권한관리, 로그관리 등)이 정상 동작하도록 통합 구현  
- 지정판매소 위치조회 서비스: 시민이 최근 지정판매소 판매내역을 조회할 수 있는 위치조회 기능  
- 보안·개발표준 준수: 전자정부표준프레임워크 도입, 오픈소스 공통서비스 활용, 시큐어 코딩 등 웹개발 보안지침 준수  
- 통계·빅데이터 연계(기대효과로 명시된 범위): 지역별·규모별·주문패턴별 분석을 통한 재고·수요 예측 지원  
- 파주시청 운영 가능성 확보: 파주시청에서도 판매관리시스템을 운영할 수 있도록 구현 및 기술이전·사용자 교육 고려

기존 데이터 이관 범위
- 기존 운영 중인 판매관리시스템의 판매소(지정판매소) 및 판매자료 일괄 이관(노후화된 기 운영 시스템의 판매소 및 판매자료 이관 명시)  

확인 근거: 파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp

[dev-multi-010] 꿀 품질평가 시스템과 원자력연 선량평가시스템의 원문상 예산을 같은 원 단위로 비교하고, 부가세 표기 차이도 알려줘.
- 한국원자력연구원 선량평가시스템 고도화: 46,600,000원 (문서에 "금 46,600 천원 (부가가치세 포함)" 명시) — 부가세 포함. 근거: 한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp

- 꿀 품질평가 전산시스템 기능개선 사업: 49,000,000원 (문서 메타데이터) — 부가세 표기 여부는 제공된 문서 범위에서는 확인되지

In [45]:
q = "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?"

hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print("현재 힌트:", hints)
print()

answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print("답변:")
print(answer)

현재 힌트: ['한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp', '한국수자원공사_건설통합시스템(CMS) 고도화.hwp', '한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp']

답변:
사업금액: 2,392,940,000원

근거: 한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp


In [46]:
remaining_orgs_tests = [
    ("한국연구재단", "한국연구재단이 대학의 산학협력 활동 실태를 조사하는 시스템 기능을 개선하는 사업의 예산은?"),
    ("국민연금공단", "국민연금공단의 온라인 학습 관련 시스템 운영 용역 기간은?"),
    ("축산물품질평가원", "축산물품질평가원이 가축의 이력을 관리하는 시스템을 개선하는 사업의 예산은?"),
    ("수협중앙회", "수협중앙회가 강릉 지역 어선의 안전을 관리하는 상황실 시스템을 구축하는 사업기간은?"),
    ("국방과학연구소", "국방과학연구소의 큰 용량 자료를 옮기는 시스템을 고도화하는 사업 예산은?"),
    ("한국생산기술연구원", "한국생산기술연구원이 진행하는 고압가스 관련 안전 시스템 구축 용역의 기간은?"),
]

for org, q in remaining_orgs_tests:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{org}] {q}")
    print(f"  힌트: {hints}")
    print()

[한국연구재단] 한국연구재단이 대학의 산학협력 활동 실태를 조사하는 시스템 기능을 개선하는 사업의 예산은?
  힌트: ['한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp', '한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp']

[국민연금공단] 국민연금공단의 온라인 학습 관련 시스템 운영 용역 기간은?
  힌트: ['국민연금공단_2024년 이러닝시스템 운영 용역.hwp', '국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp']

[축산물품질평가원] 축산물품질평가원이 가축의 이력을 관리하는 시스템을 개선하는 사업의 예산은?
  힌트: ['축산물품질평가원_축산물이력관리시스템 개선(정보화 사업).hwp', '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp']

[수협중앙회] 수협중앙회가 강릉 지역 어선의 안전을 관리하는 상황실 시스템을 구축하는 사업기간은?
  힌트: ['수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp', '수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp']

[국방과학연구소] 국방과학연구소의 큰 용량 자료를 옮기는 시스템을 고도화하는 사업 예산은?
  힌트: ['국방과학연구소_대용량 자료전송시스템 고도화.hwp', '국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp']

[한국생산기술연구원] 한국생산기술연구원이 진행하는 고압가스 관련 안전 시스템 구축 용역의 기간은?
  힌트: ['한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp', '한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp']



In [47]:
final_tests = [
    "한국연구재단이 대학의 산학협력 활동 실태를 조사하는 시스템 기능을 개선하는 사업의 예산은?",
    "국민연금공단의 온라인 학습 관련 시스템 운영 용역 기간은?",
    "축산물품질평가원이 가축의 이력을 관리하는 시스템을 개선하는 사업의 예산은?",
    "수협중앙회가 강릉 지역 어선의 안전을 관리하는 상황실 시스템을 구축하는 사업기간은?",
    "국방과학연구소의 큰 용량 자료를 옮기는 시스템을 고도화하는 사업 예산은?",
    "한국생산기술연구원이 진행하는 고압가스 관련 안전 시스템 구축 용역의 기간은?",
]

for q in final_tests:
    print(f"{q}")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

한국연구재단이 대학의 산학협력 활동 실태를 조사하는 시스템 기능을 개선하는 사업의 예산은?
129,300,000원(부가세 포함). 근거: 한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp

국민연금공단의 온라인 학습 관련 시스템 운영 용역 기간은?
계약체결일로부터 2025년 2월까지입니다. 근거: 국민연금공단_2024년 이러닝시스템 운영 용역.hwp

축산물품질평가원이 가축의 이력을 관리하는 시스템을 개선하는 사업의 예산은?
196,000,000원 (196백만원, VAT 포함). 근거: 축산물품질평가원_축산물이력관리시스템 개선(정보화 사업).hwp

수협중앙회가 강릉 지역 어선의 안전을 관리하는 상황실 시스템을 구축하는 사업기간은?
사업기간: 계약일로부터 2025년 10월 31일까지. 근거: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp

국방과학연구소의 큰 용량 자료를 옮기는 시스템을 고도화하는 사업 예산은?
316,800,000원 — 근거: 문서 '국방과학연구소_대용량 자료전송시스템 고도화.hwp' (메타데이터).

한국생산기술연구원이 진행하는 고압가스 관련 안전 시스템 구축 용역의 기간은?
확인되지 않습니다. 제공된 문서 조각에서는 고압가스 안전관리 시스템 구축 용역의 계약기간(사업기간)이 명시되어 있지 않습니다. (문서: 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp)



In [48]:
doc_id = '한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp'
doc_chunks_this = [c for c in child_chunks if c.doc_id == doc_id]

for c in doc_chunks_this:
    if '기간' in c.text and ('용역' in c.text or '사업' in c.text):
        print(c.text[:400])
        print("---")

1. 사업 개요5가. 추진배경 및 필요성5나. 사업범위5다. 기대효과52. 시스템 현황6가. 대상시스템6나. 시스템현황63. 사업 추진방안10가. 추진목표10나. 추진방안10다. 추진체계11라. 추진일정124. 제안요청 내용13가. 제안요청 개요13나. 상세요구사항191) 기능 요구사항192) 성능 요구사항243) 인터페이스 요구사항264) 데이터 요구사항365) 테스트 요구사항376) 보안 요구사항447) 품질 요구사항578) 제약사항609) 프로젝트관리 요구사항6810) 프로젝트지원 요구사항77다. 정보기술아키텍처 적용방안81라. 기타사항825. 제안서 작성요령86가. 제안서의 효력86나. 제안서 작성 시 유의사항86다. 제안서 목차88라. 세부 작성지침896. 제안안내 사항91가. 입찰방식91나. 제안
---
. 사업수행자의 준수사항99라.SW산출물 반출절차 등100마. 계약 관련사항101바. 소프트웨어사업 영향평가 검토결과서103사. 과업심의위원회 결과104아. 소프트웨어 사업 적정기간 산정결과105자. 정보누출 및 보안위규 처리 기준106[붙임 1호 서식]서약서110[붙임 2호 서식]업체 일반현황111[붙임 3호 서식]제안업체 경영상태(최근 3년)112[붙임 4호 서식]동종분야 유사용역 수행실적113[붙임 5호 서식]용역 실적이행증명서114[붙임 6호 서식]사업추진 조직115[붙임 7호 서식]참여기술자 자격사항 등 요약116[붙임 8호 서식]참여인력 이력사항117[붙임 9호 서식]공동수급표준협정서(공동이행방식118[붙임 10호 서식]공동수급 합의각서121[붙임 11호 서식]하도급 승인 신청서122[붙임 12호
---
다. 추진체계
  ○ 구성도

[표]
총  괄경영기획본부장사업지원사업관리안전보건총괄실디지털행정추진실사업수행용역업체

  ○ 역할

[표]
조직/기관 | 주 요 역 할
한국생산기술연구원 | 디지털행정추진실 | - 사업계획 및 제안요청서 작성 등 추진- 요구사항 분석 및 설계- 사업관리- 검수 및 산출물 인수
안전보건총괄실 | - 사업계약, 

In [50]:
answer_generation_results_40 = []

for item in core40:
    task_type = item['task_type']
    question = item['question']
    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    answer_generation_results_40.append({'case_id': item['case_id'], 'task_type': task_type, 'answer': answer})
    print(f"[{item['case_id']}][{task_type}] {question}")
    print(answer)
    print()

[dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 243,000,000원 — 부가가치세(VAT) 포함.  
근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp

[dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)  
예산: 70,000,000원 (VAT 포함)

근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp

[dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차로 구분됨: 1차(시스템 구축 및 초기 데이터 구축), 2차(리포팅 툴 S/W 및 리포트 출력양식 개발).  
- 평가비중: 기술평가 90%, 가격평가 10%.

근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp

[dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
사업기간: 계약일로부터 90일(문서에는 계약일로부터 3개월, 2024년 11월 1일까지라고도 표기됨).  
시범 도입 규모: 1단계 3개 기관(서울 2개소, 울산 1개소).  

근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입.hwp

[dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출방법: 입찰서와 제안서는 모두 전자적으로 제출해야 하며(나라장터 전자입찰), 입찰서와 제안서를 모두 제출해야 유효합니다. (입찰서는 나라장터에 의해 전자적으로만 제출)  
  기타 제출서류는 나라장터(e-발주시스템)를 통해 전자적

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
이 기능은 제가 수행할 수 없습니다. 실시간(오늘) 나라장터 공고 목록은 제공된 문서 범위에서 확인할 수 없습니다.  
근거: 제공된 문서(한국사회보장정보원_라오스…, 인천광역시_인천일자리플랫폼…, 한국농어촌공사_아세안+3…, 등 컨텍스트 문서들).

[dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

제공된 문서 범위에서는 최종 낙찰업체 및 낙찰금액(낙찰자 정보)이 기재되어 있지 않습니다. 원문 전체 또는 별도의 낙찰공고 확인이 필요합니다.

근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다. (실시간 전화 연결·통화는 불가합니다)

근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp)

[dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
이 요청은 귀사의 구체적 서류·현황을 대조해 판단해야 하므로 제가 낙찰자격 충족 여부를 판정할 수 없습니다. 다만 판단에 필요한 기준(문서 근거 항목)과, 귀사가 확인·제공해야 할 증빙목록은 아래와 같습니다.

판단 불가 안내
- 제시된 문서 범위만으로 귀사의 자격 충족 여부를 판정할 수 없습니다. (요건 대조는 귀사의 실자료 확인이 필요합니다.) — 근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp

평가에 필요한 자격요건(문서 근거 항목) 및 귀사가 확인해야 할 증빙 목록
1. 부정당업자 해당 여부
   - 요건: 「지방자치단체를 당사자로 하는 계약에 관한 법률 시행령 제92조」에 해당되지 않을 것.
   - 필요증빙: 관련 법 위반·처분 이력 없음 확인 자료(회사 진술서/법적 이력 조회 결과).

2. 주된 영업소 소재지
   - 요건: 입찰공고일 전일부터 계약체결일까지 주된 영업소 소재지가 부산광역시일 것.
   - 필요증빙: 사업자등록상 주된 영업소 주소, 임대계약서 등 — (문서: 입찰참가자격 2.나)

3. 나라장터(G2B) 등록
   - 요건: 입찰서 제출마감일 전일까지 G2B에 소프트웨어사업자(업종코드 1468)로 입찰참가자격 등록 완료.
   - 필요증빙: G2B 등록증/화면캡처 또는 등록번호 — (문서: 입찰참가자격 2.다)

4. 기업규모 제한(참여제한)
   - 요건: 소프트웨어산업 진흥법 제48조 및 관련 지침에 따라 대기업·중견기업 및 상호출자제한기업집단소속회사가 아닐 것.
   - 필

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
이 요청은 평가위원들의 '비공개' 또는 주관적 우선순위를 추정해 답변하는 것이므로 문서에 근거 없는 판단·추측에 해당합니다. 따라서 요청하신 대로는 답변할 수 없습니다.

대신, 제공된 제안요청서들에서 명시된 '평가 항목·배점·심사절차'를 근거로 평가위원이 공식적으로 평가하는 항목들을 정리해 드립니다. (평가위원의 비공개/주관적 우선순위 자체는 문서에 없으므로 확인 불가.)

문서들에 명시된 주요 평가항목(문서별 근거)
- 전반적 평가구성: 기술능력평가 + 가격평가(비율 및 세부조건 기관별 상이) — (부산관광공사, 조선대학교 등)
  근거: 부산관광공사_제안서 평가(기술90/가격10), 조선대학교_기술80/가격20 등.

- 기술·정성 항목(주요 세부항목)
  - 전략 및 방법론(사업이해도, 추진전략) — (한국생산기술연구원, 부산관광공사)
  - 기술 및 기능(요구사항 충족, 적용기술, 보안요구사항 등) — (한국생산기술연구원, 전북 정읍시 보안지침)
  - 성능 및 품질(품질요구사항, 인터페이스 등) — (한국생산기술연구원, 부산관광공사)
  - 프로젝트관리(일정·위험관리·문서관리 등) — (한국생산기술연구원, 부산관광공사)
  - 프로젝트지원(품질보증, 시험운영, 교육, 하자보수계획) — (한국생산기술연구원)

  근거: 한국생산기술연구원_제안서 평가 항목표, 부산관광공사_정성적 평가 구성 등.

- 정량 항목(주요 항목)
  - 수행실적(최근 3년 등, 유사업종 여부·금액 기준) — (봉화군, 국민연금공단, 부산관광공사 등)
  - 경영상태 / 신용평가등급 — (봉화군, 국민연금공단, 축산물품질평가원 등)
  - 기술인력 보유상태(투입인력 자격·이력) — (부산관광공사, 수협중앙회 별지 목록)
  근거: 봉화군_수행실적·경영상태 기준, 부산관광공사_정량평가 항목표, 수협중앙회_별지 제출서류 목록 등.

- 발표·질의응답 관련 규정
  -

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
요청하신 “입찰 참여 시작일이 문서에 없으면 공개일 다음 날로 확정”이라는 임의 가정은 제공된 문서에 근거하지 않은 확정 처리를 요구하므로 수행할 수 없습니다. 문서에 없는 정보를 임의로 가정해 확정할 수 없습니다.

대신 제가 해드릴 수 있는 것:
- 제공된 문서들에서 명시된 입찰 참여 시작일(또는 관련된 날짜)을 찾아 드리겠습니다. 날짜가 명시되지 않은 문서는 “확인되지 않습니다”로 보고하겠습니다.

어떤 문서부터 확인할까요? (현재 확인 가능한 문서 목록: 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp, 수협중앙회_수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp, 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp, 한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp, 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp, 한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp, 한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp, 한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp)

근거: 제공된 문서 목록(위)

[dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
제공된 문서 범위에서는 진행률(완료 퍼센트)과 계약 지연 가능성을 판단할 근거가 없어 답변할 수 없습니다.

필요한 정보(확인 항목, 간단히):
- 현재까지 완료된 산출물 목록 및 완료일(개발/디자인/시스템 공급별)
- 계약서상 또는 과업별 상세 일정 대비 실제 진척 

In [51]:
import re

def normalize_text(t):
    return t.replace(',', '').replace(' ', '')

def normalize_dates(text):
    text = re.sub(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', r'\1.\2.\3', text)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', lambda m: f"{m.group(1)}.{int(m.group(2)):02d}.{int(m.group(3)):02d}", text)
    return text

def _text_included(fact_text, answer_text):
    fact_text = normalize_dates(fact_text)
    answer_text_n = normalize_dates(answer_text)
    answer_norm = normalize_text(answer_text_n)

    numbers = re.findall(r'\d+(?:\.\d+)?', fact_text)
    numbers = [n for n in numbers if len(n) >= 2]

    if numbers:
        all_numbers_match = True
        for num in numbers:
            if num in answer_norm:
                continue
            if len(num) == 4 and num.startswith('20'):
                if num[2:] in answer_norm:
                    continue
            num_no_zero = re.sub(r'^0+', '', num)
            if num_no_zero and num_no_zero in answer_norm:
                continue
            all_numbers_match = False
            break

        if not all_numbers_match:
            return False

        raw_words = re.split(r'[\s,·:()]+', fact_text)
        stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
        core_words = []
        for w in raw_words:
            w = w.rstrip('.,')
            if len(w) < 2:
                continue
            if re.match(r'^\d', w):
                continue
            for suf in stopwords_suffix:
                if w.endswith(suf) and len(w) > len(suf):
                    w = w[:-len(suf)]
                    break
            w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
            if len(w) >= 2:
                core_words.append(w)

        if not core_words:
            return True

        match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
        return match_count / max(len(core_words), 1) >= 0.2

    raw_words = re.split(r'[\s,·:()]+', fact_text)
    stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
    core_words = []
    for w in raw_words:
        w = w.rstrip('.,')
        if len(w) < 2:
            continue
        if re.match(r'^\d', w):
            continue
        for suf in stopwords_suffix:
            if w.endswith(suf) and len(w) > len(suf):
                w = w[:-len(suf)]
                break
        w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
        if len(w) >= 2:
            core_words.append(w)

    if not core_words:
        return True

    match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
    return match_count / max(len(core_words), 1) >= 0.4

ABSTAIN_PHRASES = ['확인되지 않습니다', '답변할 수 없', '수행할 수 없', '확인할 수 없', '판단할 수 없',
                    '판정할 수 없', '판정해줄 수 없', '계산할 수 없', '제공할 수 없', '받아들일 수 없',
                    '확정할 수 없', '알려드릴 수 없', '불가능합니다', '제공된 문서 범위에서는']

def official_score_core40(item, answer_text):
    gold = item['gold']
    decision = gold.get('decision')

    if decision == 'abstain':
        is_abstained = any(p in answer_text for p in ABSTAIN_PHRASES)
        return 100 if is_abstained else 0

    key_points = gold.get('required_key_points', [])
    if not key_points:
        return None

    included = [_text_included(kp['text'], answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

In [52]:
for r in answer_generation_results_40:
    item = next(it for it in core40 if it['case_id'] == r['case_id'])
    r['score'] = official_score_core40(item, r['answer'])
    print(f"[{r['case_id']}][{r['task_type']}] 점수: {r['score']}")

valid_scores = [r['score'] for r in answer_generation_results_40 if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

by_type = {}
for r in answer_generation_results_40:
    if r['score'] is not None:
        by_type.setdefault(r['task_type'], []).append(r['score'])
for t, scores in by_type.items():
    print(f"{t}: 평균 {sum(scores)/len(scores):.2f}/100 ({len(scores)}개)")

[dev-single-001][single_doc] 점수: 100.0
[dev-single-002][single_doc] 점수: 100.0
[dev-single-003][single_doc] 점수: 100.0
[dev-single-004][single_doc] 점수: 100.0
[dev-single-005][single_doc] 점수: 75.0
[dev-single-006][single_doc] 점수: 100.0
[dev-single-007][single_doc] 점수: 100.0
[dev-single-008][single_doc] 점수: 100.0
[dev-single-009][single_doc] 점수: 100.0
[dev-single-010][single_doc] 점수: 66.67
[dev-multi-001][multi_doc_compare] 점수: 100.0
[dev-multi-002][multi_doc_compare] 점수: 100.0
[dev-multi-003][multi_doc_compare] 점수: 100.0
[dev-multi-004][multi_doc_compare] 점수: 100.0
[dev-multi-005][multi_doc_compare] 점수: 100.0
[dev-multi-006][multi_doc_compare] 점수: 100.0
[dev-multi-007][multi_doc_compare] 점수: 75.0
[dev-multi-008][multi_doc_compare] 점수: 100.0
[dev-multi-009][multi_doc_compare] 점수: 100.0
[dev-multi-010][multi_doc_compare] 점수: 50.0
[dev-followup-001][follow_up] 점수: 100.0
[dev-followup-002][follow_up] 점수: 50.0
[dev-followup-003][follow_up] 점수: 100.0
[dev-followup-004][follow_up] 점수: 100.0
[dev

In [53]:
followup_check_ids = ['dev-followup-002', 'dev-followup-009', 'dev-followup-010']

for cid in followup_check_ids:
    item = next(it for it in core40 if it['case_id'] == cid)
    question = item['question']
    history = item.get('history', [])
    user_turns = [h['content'] for h in history if h.get('role') == 'user']
    prev_q = user_turns[-1] if user_turns else ""
    combined_q = f"{prev_q} {question}"

    scores = []
    for i in range(3):
        answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
        score = official_score_core40(item, answer)
        scores.append(score)

    has_variance = len(set(scores)) > 1
    flag = "[변동있음]" if has_variance else "[안정]"
    print(f"[{cid}] {scores} {flag}")

[dev-followup-002] [50.0, 50.0, 50.0] [안정]
[dev-followup-009] [50.0, 50.0, 100.0] [변동있음]
[dev-followup-010] [100.0, 100.0, 100.0] [안정]


In [54]:
item = next(it for it in core40 if it['case_id'] == 'dev-followup-002')
print("질문:", item['question'])
print("정답 요소:")
for kp in item['gold']['required_key_points']:
    print(f"  - {kp['text']}")
print()

history = item.get('history', [])
user_turns = [h['content'] for h in history if h.get('role') == 'user']
prev_q = user_turns[-1] if user_turns else ""
combined_q = f"{prev_q} {item['question']}"

answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print("답변:")
print(answer)

질문: 그 6개월은 차수별로 어떻게 나눠져?
정답 요소:
  - 1차는 4개월이며 시스템과 초기 데이터를 구축한다.
  - 2차는 1차 완료 후 2개월이며 리포팅툴과 출력양식을 개발한다.

답변:
사업기간: 계약일로부터 6개월.

차수별 구성:
- 1차 사업기간: 계약일로부터 4개월
- 2차 사업기간: 1차 완료일로부터 2개월

근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp


In [55]:
# rag-56에서도 같은 발주기관에 문서 여러 개인 케이스가 있는지 확인
multi_doc_orgs_check = ['한국철도공사', '한국수자원공사', '인천광역시', '광주과학기술원', '한국연구재단',
                          '한국생산기술연구원', '국방과학연구소', '수협중앙회', '한국농어촌공사',
                          '축산물품질평가원', '국민연금공단']

for item in rag56:
    q = item['question']
    for org in multi_doc_orgs_check:
        if org in q:
            print(f"[{item['case_id']}] (기관: {org}) {q}")

[supplemental-qa-c05] (기관: 한국농어촌공사) 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
[supplemental-qa-c06] (기관: 한국농어촌공사) 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성한 예산은 얼마인가요? 세금 포함 여부도 알려주세요.
[supplemental-qa-c09] (기관: 한국철도공사) 한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?
[supplemental-qa-c10] (기관: 한국수자원공사) 한국수자원공사가 건설 업무를 하나로 관리하는 시스템을 고도화하는 과업은 착수 후 얼마나 수행하나요?
[supplemental-qa-c11] (기관: 한국철도공사) 한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?
[supplemental-qa-c15] (기관: 국방과학연구소) 국방과학연구소 '기록관리시스템 통합 활용 및 보안 환경 구축' 사업에 공동수급체 소속이 아닌 외부 인력을 투입해도 되나요? 투입할 경우 하도급으로 처리되나요?
[supplemental-qa-c16] (기관: 한국철도공사) 한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?
[supplemental-qa-c18] (기관: 한국농어촌공사) 한국농어촌공사 네팔 수자원관리 정보화사업 Pilot 시스템 구축용역에 참여하려면 나라장터(G2B) 등록을 언제까지 마쳐야 하나요?
[supplemental-qa-c19] (기관: 한국철도공사) 다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및

In [56]:
new_check_ids = ['supplemental-qa-c09', 'supplemental-qa-c11', 'supplemental-qa-c15',
                  'supplemental-qa-c16', 'supplemental-qa-c18', 'supplemental-qa-c19',
                  'supplemental-qa-c23', 'supplemental-qa-g16', 'supplemental-qa-g17',
                  'supplemental-alignment-h18']

for cid in new_check_ids:
    item = next(it for it in rag56 if it['case_id'] == cid)
    q = item['question']
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{cid}] {q}")
    print(f"  힌트: {hints}")
    print()

[supplemental-qa-c09] 한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?
  힌트: ['한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']

[supplemental-qa-c11] 한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?
  힌트: ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp']

[supplemental-qa-c15] 국방과학연구소 '기록관리시스템 통합 활용 및 보안 환경 구축' 사업에 공동수급체 소속이 아닌 외부 인력을 투입해도 되나요? 투입할 경우 하도급으로 처리되나요?
  힌트: ['국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp', '국방과학연구소_대용량 자료전송시스템 고도화.hwp']

[supplemental-qa-c16] 한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?
  힌트: ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp']

[supplemental-qa-c18] 한국농어촌공사 네팔 수자원관리 정보화사업 Pilot 시스템 구축용역에 참여하려면 나라장터(G2B) 등록을 언제까지 마쳐야 하나요?
  힌트: ['한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp', '한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp', '축산물품질평가원_축산물이력관리시스템 개선(정보화 사업).hwp']

[supplemental-qa-c19] 다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업

In [57]:
q1 = "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?"
q2 = "수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업의 계약보증금 기준은 무엇인가요?"

# "개량"이라는 단어가 문제인지 확인
for q in [q1, q2]:
    print(f"질문: {q}")
    print(f"  '개량' 포함?: {'개량' in q}")
    print(f"  'ISMP' 포함?: {'ISMP' in q}")

질문: 한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?
  '개량' 포함?: True
  'ISMP' 포함?: False
질문: 수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업의 계약보증금 기준은 무엇인가요?
  '개량' 포함?: False
  'ISMP' 포함?: True


In [58]:
fname = '한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역'

def fuzzy_match(kw, text, min_overlap=6):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

q1_keywords = ['개량']
q2_keywords = ['ISMP']

for kw in q1_keywords + q2_keywords:
    result = fuzzy_match(kw, fname)
    print(f"'{kw}' -> {result}")

'개량' -> True
'ISMP' -> True


In [59]:
COMMON_FILENAME_WORDS_test = COMMON_FILENAME_WORDS | {'개량', 'ISMP'}

raw_keywords_test = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?") if len(w) >= 4]
keywords_all_test = [w for w in raw_keywords_test if w not in COMMON_FILENAME_WORDS_test]
print("c11 키워드(개량 제외 후):", keywords_all_test)

raw_keywords_test2 = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', "수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업의 계약보증금 기준은 무엇인가요?") if len(w) >= 4]
keywords_all_test2 = [w for w in raw_keywords_test2 if w not in COMMON_FILENAME_WORDS_test]
print("g16 키워드(ISMP 제외 후):", keywords_all_test2)

c11 키워드(개량 제외 후): ['한국철도공사', '운행정보기록', '자동분석시스템', '하도급이', '가능한가요']
g16 키워드(ISMP 제외 후): ['수협중앙회', '수산물사이버직매장', '계약보증금', '무엇인가요']


In [62]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}"
new_code = "COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역', '개량', 'ISMP'}"

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [63]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

importlib.reload(answer_generation)

test_cases = [
    "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?",
    "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?",
    "수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업의 계약보증금 기준은 무엇인가요?",
    "수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입찰이 재입찰 또는 재공고입찰로 진행되면 최초 조건을 변경할 수 있나요?",
    "수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업은 어떤 사업인가요?",
]

for q in test_cases:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?]
  힌트: ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp']

[한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?]
  힌트: ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp']

[수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업의 계약보증금 기준은 무엇인가요?]
  힌트: ['수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp', '수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp']

[수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입찰이 재입찰 또는 재공고입찰로 진행되면 최초 조건을 변경할 수 있나요?]
  힌트: ['수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp', '수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp']

[수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업은 어떤 사업인가요?]
  힌트: ['수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp', '수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp']



In [64]:
q = "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?"

stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', q) if len(w) >= 4]
keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]
print("키워드:", keywords_all)

def fuzzy_match(kw, text, min_overlap=6):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

for fname in ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역', '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스']:
    matched = [kw for kw in keywords_all if fuzzy_match(kw, fname)]
    print(f"{fname}: matched={matched}")

키워드: ['한국철도공사', '운행정보기록', '자동분석시스템', '하도급이', '가능한가요']
한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역: matched=['한국철도공사']
한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스: matched=['한국철도공사', '운행정보기록', '자동분석시스템']


In [65]:
q = "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?"

# 4단계 로직 직접 재현
stopwords_4 = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교',
               '용역은', '용역이', '용역을', '사업은', '사업이', '사업을'}
keywords_4 = [w for w in re.split(r'[ ,]', q) if len(w) >= 2 and w not in stopwords_4]
print("4단계 키워드:", keywords_4)

def _fuzzy_kw_match(kw, text):
    kw_clean = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', kw)
    if len(kw_clean) < 2:
        return False
    return kw_clean in text.replace(' ', '')

for fname in ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp', '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']:
    matched = [kw for kw in keywords_4 if _fuzzy_kw_match(kw, fname)]
    print(f"{fname}: score={len(matched)}, matched={matched}")

4단계 키워드: ['한국철도공사', '운행정보기록', '자동분석시스템', '개량', '하도급이', '가능한가요?']
한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp: score=2, matched=['한국철도공사', '개량']
한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp: score=2, matched=['한국철도공사', '운행정보기록']


In [66]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

# 파일 안 실제 4단계 stopwords 텍스트 확인
idx = content.find("stopwords = {'사업의'")
print(content[idx:idx+250])

stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교',
                 '용역은', '용역이', '용역을', '사업은', '사업이', '사업을'}
    keywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords]

    def _fuzzy_kw_match


In [67]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교',\n                 '용역은', '용역이', '용역을', '사업은', '사업이', '사업을'}"
new_code = "    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교',\n                 '용역은', '용역이', '용역을', '사업은', '사업이', '사업을', '개량'}"

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [68]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

importlib.reload(answer_generation)

test_cases = [
    "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?",
    "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?",
]

for q in test_cases:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?]
  힌트: ['한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']

[한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?]
  힌트: ['한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']



In [69]:
q = "한국농어촌공사 네팔 수자원관리 정보화사업 Pilot 시스템 구축용역에 참여하려면 나라장터(G2B) 등록을 언제까지 마쳐야 하나요?"

org_part = '축산물품질평가원'
org_core_clean = org_part
org_core_norm = normalize_org_name(org_core_clean)

min_len = 6
for target_str in [org_core_clean, org_core_norm]:
    for start in range(len(target_str) - min_len + 1):
        for length in range(len(target_str) - start, min_len - 1, -1):
            substr = target_str[start:start+length]
            stripped = substr.strip()
            if len(stripped) >= min_len and stripped in q and stripped not in COMMON_SUFFIX_WORDS:
                print(f"매칭됨(1단계): '{stripped}'")

# 3단계도 확인
stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', q) if len(w) >= 4]
keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]

def fuzzy_match(kw, text, min_overlap=6):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

fname = '축산물품질평가원_축산물이력관리시스템 개선(정보화 사업)'
matched = [kw for kw in keywords_all if fuzzy_match(kw, fname)]
print(f"3단계 매칭: {matched}")

3단계 매칭: ['정보화사업']


In [70]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역', '개량', 'ISMP'}"
new_code = "COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역', '개량', 'ISMP', '정보화사업', '정보화'}"

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [71]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

importlib.reload(answer_generation)

test_cases = [
    "한국농어촌공사 네팔 수자원관리 정보화사업 Pilot 시스템 구축용역에 참여하려면 나라장터(G2B) 등록을 언제까지 마쳐야 하나요?",
    "한국농어촌공사 네팔 수자원관리 정보화사업 Pilot 시스템 구축용역의 계약이행보증금 비율이 문서에 명시되어 있나요? 명시되어 있다면 몇 퍼센트인가요?",
]

for q in test_cases:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[한국농어촌공사 네팔 수자원관리 정보화사업 Pilot 시스템 구축용역에 참여하려면 나라장터(G2B) 등록을 언제까지 마쳐야 하나요?]
  힌트: ['한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp', '한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp']

[한국농어촌공사 네팔 수자원관리 정보화사업 Pilot 시스템 구축용역의 계약이행보증금 비율이 문서에 명시되어 있나요? 명시되어 있다면 몇 퍼센트인가요?]
  힌트: ['한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp', '한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp']



In [72]:
q = "다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다."
hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print(hints)

['국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp', '한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp', '한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp', '인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp', '그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp', '국방과학연구소_대용량 자료전송시스템 고도화.hwp', '한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp', '고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', '한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp', '경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp', '사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp', '한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp']


In [73]:
q = "다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다."

# 4단계 재확인 - "한국철도공사"로 묶인 org_group에서 어느게 이기는지
stopwords_4 = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교',
               '용역은', '용역이', '용역을', '사업은', '사업이', '사업을', '개량'}
keywords_4 = [w for w in re.split(r'[ ,]', q) if len(w) >= 2 and w not in stopwords_4]

def _fuzzy_kw_match(kw, text):
    kw_clean = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', kw)
    if len(kw_clean) < 2:
        return False
    return kw_clean in text.replace(' ', '')

for fname in ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp', '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']:
    matched = [kw for kw in keywords_4 if _fuzzy_kw_match(kw, fname)]
    print(f"{fname}: score={len(matched)}, matched={matched}")

한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp: score=2, matched=['한국철도공사', '시스템']
한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp: score=2, matched=['한국철도공사', '운행정보기록']


In [74]:
def _fuzzy_kw_match(kw, text):
    kw_clean = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', kw)
    if len(kw_clean) < 2:
        return False
    return kw_clean in text.replace(' ', '')

# "시스템"만 2글자인데 매칭 조건 확인
print(_fuzzy_kw_match('시스템', '한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp'))

True


In [75]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교',\n                 '용역은', '용역이', '용역을', '사업은', '사업이', '사업을', '개량'}"
new_code = "    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교',\n                 '용역은', '용역이', '용역을', '사업은', '사업이', '사업을', '개량', '시스템', '시스템은', '시스템이'}"

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [76]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

importlib.reload(answer_generation)

q = "다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다."
hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print(hints)

['국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp', '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp', '한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp', '인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp', '그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp', '국방과학연구소_대용량 자료전송시스템 고도화.hwp', '한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp', '고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', '한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp', '경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp', '사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp', '한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp']


In [77]:
answer_generation_final_40 = []
for item in core40:
    task_type = item['task_type']
    question = item['question']
    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    answer_generation_final_40.append({'case_id': item['case_id'], 'task_type': task_type, 'answer': answer})
    print(f"[{item['case_id']}][{task_type}] {question}")
    print(answer)
    print()

[dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 금 243,000,000원 — 부가가치세(VAT) 포함. 근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp

[dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)  
예산: 70,000,000원(₩70,000,000, VAT 포함)

근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp

[dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차로 구분됨: 1차(계약일로부터 4개월), 2차(1차 완료일로부터 2개월).  
- 평가비중: 기술평가 90%, 가격평가 10%.

근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp

[dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
- 사업기간: 계약일로부터 90일(문서에 3개월로도 표기; 완료 목표일 예시: 2024년 11월 1일까지) — 근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입.hwp

- 시범 도입 규모: 1단계 3개 기관(서울 2개소, 울산 1개소) — 근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입.hwp

[dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출방법: 입찰서와 제안서는 전자적으로 제출해야 하며(입찰서는 나라장터 전자제출만 인정). 기타 제출서류는 나라장터(e-발주시스템)를 통해 전자 제출하

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
실시간(오늘)으로 새로 올라온 나라장터 공고 조회는 제가 수행할 수 없습니다. 제공된 문서들에는 '오늘' 게시된 공고 목록이 포함되어 있지 않습니다. 근거: 제공된 문서들(한국사회보장정보원_라오스…, 인천광역시_인천일자리플랫폼…, 한국농어촌공사_아세안+3…, 2025 구미 아시아육상경기…, 한국생산기술연구원_2세대 전자조달…, 서민금융진흥원_서민금융 채팅…, 중앙선거관리위원회_2025년도 행정정보시스템…, 전북특별자치도 정읍시_정읍체육트레이닝센터 …)

[dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

근거 문서: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다. 근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp)

[dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
요청하신 “자사 자격 충족 여부 판정”은 사용자의 내부 정보(사업자 소재지, 나라장터 등록상태, 중소기업 여부 등)와 문서의 내용을 대조해 주관적으로 판단해야 하므로 제공된 문서만으로 판정할 수 없습니다. 따라서 판정해 드릴 수 없습니다.

대신 판정에 필요한 확인 항목(문서에 근거)을 정리합니다 — 아래 항목들을 귀사 내부 자료로 확인해 주세요.

필수 확인 항목(근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp)
1. 부정당업자 해당 여부
   - 「지방자치단체를 당사자로 하는 계약에 관한 법률 시행령」 제92조에 해당하지 않는지 확인.
2. 주된 영업소 소재지
   - 입찰공고일 전일부터 계약체결일까지 주된 영업소 소재지가 부산광역시인지 확인(동법 시행령 제20조 제1항 6호 및 시행규칙 제24조 제2항 근거).
3. 나라장터(G2B) 입찰참가자격 등록
   - 입찰서 제출마감일 전일까지 나라장터에 소프트웨어사업자(업종코드 1468, 컴퓨터관련서비스사업)로 등록되어 있는지 확인.
4. 기업 유형(대기업·중견기업 제한)
   - 소프트웨어산업 진흥법 및 관련 지침에 따라 대기업·중견기업 또는 상호출자제한기업집단 소속회사가 아닌지 확인(참여 불가 규정).
5. 직접생산확인증명서
   - 중소기업제품 구매촉진법 규정에 따른 정보시스템개발서비스(세부품명번호 8111159901)의 ‘직접생산확인증명서’를 입찰마감 전일까지 발급받아 유효한지 확인.
6. 공동수급·하도급 여부
   - 공동수급(공동이행방식) 및 하도급이 허용되지 않으므로, 단독으로 수행 가능하

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
확답할 수 없습니다.

이 문서들에는 평가위원들이 “비공개로 더 중요하게 보는 포인트(심사위원 개인의 비공식적 판단 기준)”에 대한 정보가 포함되어 있지 않거나 공개되지 않았습니다. 또한 제시된 문서들(예: 수협중앙회, 조선대학교 등)은 평가 세부결과·평가위원 명단·평가내용을 비공개로 한다고 명시하고 있어, 문서에 근거 없는 추측이나 판단은 제공할 수 없습니다.

대신 문서에 명시된 공개된 평가항목·배점·평가운영방식 등은 아래 섹션에서 확인할 수 있습니다. 실제 평가에서 참고할 수 있는 근거로는 해당 문서의 ‘제안평가 방법’, ‘평가기준 및 배점’, ‘제출서류·증빙 요건’, ‘발표 및 질의응답 운영’ 등을 검토하시기를 권합니다.

근거 문서:
- 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp (제안평가회 및 평가방법, 평가결과 비공개 명시)
- 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp (평가 비공개 및 평가방식)
- 부산관광공사_경영정보시스템 기능개선.hwp (평가비율·세부배점 등 공개된 평가기준)

원하시면 위 공개된 평가항목·배점·발표요건 등을 문서별로 정리해 드리겠습니다.

[dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
요청하신 '수주확률을 퍼센트로 계산'하는 것은 제공된 문서만으로 판단·확률을 산정할 수 없어 답변할 수 없습니다.

대신 수주 가능성 평가에 실제로 필요한 주요 항목들(귀사가 비교·판단할 때 확인해야 할 조건)을 간단히 제시합니다.
- 입찰참가자격 충족 여부(지역제한, 필수 자격 등) — BIFF&ACFM: 제한 경쟁입찰(지역 제한) 등 관련 조항 확인필요.  
- 가격(제안 금액) 및 예산과의 적합성 — BIFF&ACFM: 사업예산 243,000,000원(VA

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
요청대로 임의 규칙을 적용해 날짜를 확정할 수 없습니다. 문서에 없는 정보는 임의로 가정하여 확정 답변할 수 없습니다. 원문에서 입찰 참여 시작일을 찾아드리거나, 특정 문서의 공개일을 알려주시면 그 문서 범위 내에서 확인해 드리겠습니다.

근거: 대화에 제공된 기본 원칙 (문서에 없는 정보는 임의 가정 금지)

[dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
답변할 수 없습니다.

- 사업의 현재 진행률(몇 % 진행됐는지)은 제공된 문서에 명시되어 있지 않아 확인할 수 없습니다. (확인되지 않습니다.)
- 계약 지연 가능성에 대한 판단(예상·확률)은 제안요청서 문서만으로는 평가할 수 없으므로 답변할 수 없습니다.

대신 판단에 필요한 주요 근거 항목(참고용)은 아래와 같습니다. 이 항목들이 문서나 발주기관으로부터 확인되어야 계약 지연 리스크를 평가할 수 있습니다.
- 현재 산출물(설계서, 시안, 개발진척 보고서 등)의 제출·승인 상태 및 일정 대비 실제 진척률
- 계약 체결(실제 계약서 서명) 여부 및 체결 일자
- 주요 마일스톤(개발·테스트·인도)별 완료 예정일과 실제 수행 일정
- 인력 파견·자원 투입 여부(파견인원, 개시일)
- 외부 연동(티켓·채용 등) 또는 장비(서버/CDN, 바코드리더 등) 조달/구현상의 이슈 발생 여부
- 예산 집행 현황 및 변경·추가 요구(범위 변경) 여부
- 발주기관의 승인·결재 지연 사례 유무

근거 문서: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp



In [78]:
for r in answer_generation_final_40:
    item = next(it for it in core40 if it['case_id'] == r['case_id'])
    r['score'] = official_score_core40(item, r['answer'])
    print(f"[{r['case_id']}][{r['task_type']}] 점수: {r['score']}")

valid_scores = [r['score'] for r in answer_generation_final_40 if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

by_type = {}
for r in answer_generation_final_40:
    if r['score'] is not None:
        by_type.setdefault(r['task_type'], []).append(r['score'])
for t, scores in by_type.items():
    print(f"{t}: 평균 {sum(scores)/len(scores):.2f}/100 ({len(scores)}개)")

[dev-single-001][single_doc] 점수: 100.0
[dev-single-002][single_doc] 점수: 100.0
[dev-single-003][single_doc] 점수: 100.0
[dev-single-004][single_doc] 점수: 100.0
[dev-single-005][single_doc] 점수: 75.0
[dev-single-006][single_doc] 점수: 100.0
[dev-single-007][single_doc] 점수: 100.0
[dev-single-008][single_doc] 점수: 100.0
[dev-single-009][single_doc] 점수: 100.0
[dev-single-010][single_doc] 점수: 66.67
[dev-multi-001][multi_doc_compare] 점수: 100.0
[dev-multi-002][multi_doc_compare] 점수: 100.0
[dev-multi-003][multi_doc_compare] 점수: 100.0
[dev-multi-004][multi_doc_compare] 점수: 100.0
[dev-multi-005][multi_doc_compare] 점수: 75.0
[dev-multi-006][multi_doc_compare] 점수: 100.0
[dev-multi-007][multi_doc_compare] 점수: 75.0
[dev-multi-008][multi_doc_compare] 점수: 100.0
[dev-multi-009][multi_doc_compare] 점수: 100.0
[dev-multi-010][multi_doc_compare] 점수: 50.0
[dev-followup-001][follow_up] 점수: 100.0
[dev-followup-002][follow_up] 점수: 50.0
[dev-followup-003][follow_up] 점수: 100.0
[dev-followup-004][follow_up] 점수: 100.0
[dev-

In [79]:
answer_generation_final_56 = []
for item in rag56:
    question = item['question']
    answer = ask_rfp_v9(question, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    answer_generation_final_56.append({'case_id': item['case_id'], 'answer': answer})
    print(f"[{item['case_id']}] {question}")
    print(answer)
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c01] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원(부가세 포함). 근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c02] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024.10.31.까지 완료해야 합니다. 근거: 2024년도 평택시 버스정보시스템(BIS) 구축사업 용역 (문서).

[supplemental-qa-c03] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
사업예산: 1,515,000천원 (부가세 포함). 근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

[supplemental-qa-c04] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
경쟁 방식: 제한경쟁입찰  
낙찰 절차(사업자 선정방식): 협상에 의한 계약(「국가를 당사자로 하는 계약에 관한 법률 시행령」 및 관련 기준에 따름)

근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

[supplemental-qa-c05] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월입니다. 근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp

[supplemental-qa-c06] 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성한 예산은 얼마인가요? 세금 포함 여부도 알려주세요.
사업예산은 181,913,000원이며 VAT 포함 표기입니다. 근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp

[supplemental-qa-c07] 인천공항운영서비스가 회계·인사 등 경영업무를 통합할 차세대 시스템을 구축하는 데 걸리는 기간은 얼마인가요?
계약 체결일로부터 9개월(안정화 기간 포함)입니다. 근거: 인천공항운영서비스㈜ 차세대 ERP시스템 구축 제안요청서 (.hwp)

In [80]:
def check_required_facts(answer_text, required_fact_groups):
    if not required_fact_groups:
        return 0, 0
    matched = 0
    for group in required_fact_groups:
        if any(fact in answer_text for fact in group):
            matched += 1
    return matched, len(required_fact_groups)

In [83]:
for r in answer_generation_final_56:
    item = next(it for it in rag56 if it['case_id'] == r['case_id'])
    matched, total = check_required_facts(r['answer'], item['gold'].get('required_fact_groups'))
    r['score'] = round(matched / total * 100, 2) if total else None
    print(f"[{r['case_id']}] 점수: {r['score']}")

valid_scores_56 = [r['score'] for r in answer_generation_final_56 if r['score'] is not None]
print(f"\nrag-56 전체 평균: {sum(valid_scores_56)/len(valid_scores_56):.2f}/100 ({len(valid_scores_56)}개)")

[supplemental-qa-c01] 점수: 100.0
[supplemental-qa-c02] 점수: 100.0
[supplemental-qa-c03] 점수: 100.0
[supplemental-qa-c04] 점수: 100.0
[supplemental-qa-c05] 점수: 100.0
[supplemental-qa-c06] 점수: 100.0
[supplemental-qa-c07] 점수: 100.0
[supplemental-qa-c08] 점수: 100.0
[supplemental-qa-c09] 점수: 100.0
[supplemental-qa-c10] 점수: 100.0
[supplemental-qa-c11] 점수: 100.0
[supplemental-qa-c12] 점수: 66.67
[supplemental-qa-c13] 점수: 100.0
[supplemental-qa-c14] 점수: 100.0
[supplemental-qa-c15] 점수: 100.0
[supplemental-qa-c16] 점수: 50.0
[supplemental-qa-c18] 점수: 0.0
[supplemental-qa-c19] 점수: 0.0
[supplemental-qa-c20] 점수: 0.0
[supplemental-qa-c23] 점수: 0.0
[supplemental-qa-c25] 점수: 50.0
[supplemental-qa-g01] 점수: 100.0
[supplemental-qa-g02] 점수: 100.0
[supplemental-qa-g03] 점수: 50.0
[supplemental-qa-g04] 점수: 100.0
[supplemental-qa-g05] 점수: 100.0
[supplemental-qa-g06] 점수: 100.0
[supplemental-qa-g07] 점수: 100.0
[supplemental-qa-g08] 점수: 0.0
[supplemental-qa-g11] 점수: 33.33
[supplemental-qa-g12] 점수: 100.0
[supplemental-qa-g13]

In [84]:
final_verification_cases = [
    ("한국철도공사(c09)", "한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?", "운행정보기록 자동분석시스"),
    ("한국철도공사(c11)", "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?", "운행정보기록 자동분석시스"),
    ("한국철도공사(c16)", "한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?", "운행정보기록 자동분석시스"),
    ("인천광역시", "인천광역시가 발주한 도시계획위원회 통합관리시스템은 사업기간이 얼마나 되나요?", "인천광역시_도시계획위원회"),
    ("광주과학기술원", "광주과학기술원의 학생 성적/수강 관련 시스템을 고치는 사업기간은?", "광주과학기술원"),
    ("한국수자원공사", "한국수자원공사가 용인 반도체 산업단지에 물을 공급하는 사업의 타당성 조사 예산은?", "용인 첨단 시스템반도체"),
    ("한국수자원공사(c25)", "한국수자원공사 건설통합시스템(CMS) 고도화 사업의 제안요청서에 총 사업예산이 명시되어 있나요? 명시되어 있다면 얼마인가요?", "건설통합시스템(CMS)"),
    ("한국농어촌공사(c18)", "한국농어촌공사 네팔 수자원관리 정보화사업 Pilot 시스템 구축용역에 참여하려면 나라장터(G2B) 등록을 언제까지 마쳐야 하나요?", "네팔 수자원관리"),
    ("한국농어촌공사(c23)", "한국농어촌공사 네팔 수자원관리 정보화사업 Pilot 시스템 구축용역의 계약이행보증금 비율이 문서에 명시되어 있나요? 명시되어 있다면 몇 퍼센트인가요?", "네팔 수자원관리"),
    ("수협중앙회(g16)", "수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업의 계약보증금 기준은 무엇인가요?", "수산물사이버직매장"),
    ("수협중앙회(g17)", "수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입찰이 재입찰 또는 재공고입찰로 진행되면 최초 조건을 변경할 수 있나요?", "수산물사이버직매장"),
    ("수협중앙회(h18)", "수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업은 어떤 사업인가요?", "수산물사이버직매장"),
    ("국방과학연구소(c15)", "국방과학연구소 '기록관리시스템 통합 활용 및 보안 환경 구축' 사업에 공동수급체 소속이 아닌 외부 인력을 투입해도 되나요? 투입할 경우 하도급으로 처리되나요?", "기록관리시스템 통합"),
]

print(f"검증 대상: {len(final_verification_cases)}개")
print()

all_pass = True
for label, q, expected_keyword in final_verification_cases:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    top_hint = hints[0] if hints else None
    is_correct = top_hint is not None and expected_keyword in top_hint
    status = "통과" if is_correct else "실패"
    if not is_correct:
        all_pass = False
    print(f"[{status}] {label}")
    print(f"   1순위 힌트: {top_hint}")
    print(f"   전체 힌트: {hints}")
    print()

print(f"결과: {'전체 통과' if all_pass else '일부 실패'}")

검증 대상: 13개

[통과] 한국철도공사(c09)
   1순위 힌트: 한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp
   전체 힌트: ['한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']

[통과] 한국철도공사(c11)
   1순위 힌트: 한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp
   전체 힌트: ['한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']

[통과] 한국철도공사(c16)
   1순위 힌트: 한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp
   전체 힌트: ['한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']

[통과] 인천광역시
   1순위 힌트: 인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp
   전체 힌트: ['인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp', '인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp', '경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp']

[통과] 광주과학기술원
   1순위 힌트: 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp
   전체 힌트: ['광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp', '광주과학기술원_학사시스템 기능개선 사업.hwp']

[통과] 한국수자원공사
   1순위 힌트: 한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp
   전체 힌트: ['한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp', '한국수자원공사_건설통합시스템(CMS) 고도화.hwp', '한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp']

[통과] 한국수자원공사(c25)
   1순위 힌트: 한국수자원공사_건설통합시스템(CMS) 고도화.hwp
  

In [85]:
# 전체 98개 문서 파일명에서 자주 등장하는 단어 빈도 확인

word_counter = Counter()
for fname, biz in all_filenames_with_biz:
    fname_clean = fname.replace('.hwp', '').replace('.pdf', '')
    words = re.split(r'[\s_·\(\)]+', fname_clean)
    for w in words:
        w = w.strip()
        if len(w) >= 4:
            word_counter[w] += 1

# 여러 문서 파일명에 등장하는 단어들 (오탐 위험군)
risky_words = {w: c for w, c in word_counter.items() if c >= 3}
print(f"3개 이상 문서에 등장하는 4글자+ 단어: {len(risky_words)}개")
for w, c in sorted(risky_words.items(), key=lambda x: -x[1]):
    print(f"  '{w}': {c}회")

3개 이상 문서에 등장하는 4글자+ 단어: 12개
  '2024년': 13회
  '기능개선': 7회
  '정보시스템': 7회
  '홈페이지': 4회
  '인천광역시': 3회
  '구축사업': 3회
  '한국수자원공사': 3회
  '한국철도공사': 3회
  '수협중앙회': 3회
  '2025년': 3회
  '전산시스템': 3회
  '재단법인': 3회


In [86]:
# 위험 단어들이 실제로 서로 다른 발주기관 문서를 오염시키는지 확인
risky_test_words = ['기능개선', '정보시스템', '홈페이지', '구축사업', '전산시스템']

for word in risky_test_words:
    matching_docs = [f for f, biz in all_filenames_with_biz if word in f]
    print(f"'{word}' 포함 문서 ({len(matching_docs)}개):")
    for d in matching_docs:
        print(f"  {d}")
    print()

'기능개선' 포함 문서 (7개):
  한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp
  재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp
  울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp
  부산관광공사_경영정보시스템 기능개선.hwp
  광주과학기술원_학사시스템 기능개선 사업.hwp
  축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp
  한국보육진흥원_연차별 자율 품질관리 시스템 기능개선.hwp

'정보시스템' 포함 문서 (19개):
  고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
  한국사학진흥재단_대학재정정보시스템(기본재산 및 기채 사후관리) 고.hwp
  경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
  한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp
  한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp
  인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp
  울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp
  부산관광공사_경영정보시스템 기능개선.hwp
  한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
  국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp
  중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp
  한국한의학연구원_통합정보시스템 고도화 용역.hwp
  경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp
  한국건강가정진흥원_2025년 아이돌봄인력 인적성 검사 정보시스템 운영.hwp
  남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp
  재단법인 한국장애인문화예술원_2024년 장애인문화예술정보시스템 이음.hwp
  경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp
  한국산업단지공단_산단 안전정보시스